# Principles of Natural Language Processing Lab  
## Language Modelling: N-grams, N-gram Probabilities, Evaluation and Perplexity

**Course:** AME 5053 — Principles of Natural Language Processing Lab  
**Lab duration:** 3 hours  
**Mode:** Guided implementation + comparison + interpretation

### Learning outcomes
By the end of this lab, you should be able to:

1. construct unigram, bigram and trigram language models from a text corpus;
2. estimate N-gram probabilities using maximum-likelihood estimation (MLE);
3. compute the probability of a sentence under an N-gram model;
4. explain the effect of sentence-boundary symbols such as `<s>` and `</s>`;
5. evaluate a language model on unseen text using perplexity;
6. diagnose why an unsmoothed N-gram model can fail on unseen N-grams.

### What you must submit
Your notebook should contain:
- your predictions before running selected cells;
- completed implementations;
- at least one comparison between models;
- an error analysis using concrete examples;
- a short conclusion explaining what model behaviour you observed.

## 1. Corpus used in this lab

We will use a small, controlled corpus so that the counts can be inspected manually.

```text
students learn natural language processing
students learn machine learning
students study language models
language models predict words
language models assign probabilities
machine learning models learn patterns
natural language processing uses language models
```

The corpus is intentionally small. This makes the probability calculations transparent, but it will also expose an important limitation: **many valid word sequences will never occur in the training corpus**.

We will later split examples into **training** and **test** sentences so that perplexity is computed on data that the model did not simply memorize.

In [1]:
from collections import Counter, defaultdict
import math
import pandas as pd

raw_sentences = [
    "students learn natural language processing",
    "students learn machine learning",
    "students study language models",
    "language models predict words",
    "language models assign probabilities",
    "machine learning models learn patterns",
    "natural language processing uses language models",
]

raw_sentences

['students learn natural language processing',
 'students learn machine learning',
 'students study language models',
 'language models predict words',
 'language models assign probabilities',
 'machine learning models learn patterns',
 'natural language processing uses language models']

## 1.1 Tokenize the corpus
Run the helper below and inspect the output.

In [2]:
def tokenize(sentence):
    return sentence.lower().split()

tokenized = [tokenize(s) for s in raw_sentences]
tokenized

[['students', 'learn', 'natural', 'language', 'processing'],
 ['students', 'learn', 'machine', 'learning'],
 ['students', 'study', 'language', 'models'],
 ['language', 'models', 'predict', 'words'],
 ['language', 'models', 'assign', 'probabilities'],
 ['machine', 'learning', 'models', 'learn', 'patterns'],
 ['natural', 'language', 'processing', 'uses', 'language', 'models']]

## 2. Sentence boundaries

A language model should know not only which words can follow other words, but also where a sentence can begin and end.

For a bigram model we will represent:

```text
students learn machine learning
```

as

```text
<s> students learn machine learning </s>
```

For a trigram model, we need enough history at the beginning. We therefore use two start symbols:

```text
<s> <s> students learn machine learning </s>
```

This convention allows the model to estimate probabilities for sentence beginnings as well as sentence endings.

### Prediction 1 — before coding

Without running any code, answer:

1. Which word do you expect to be the most frequent unigram?
2. Which bigram do you expect to have a relatively high probability?
3. Do you expect a trigram model to assign non-zero probability to more or fewer unseen test sentences than a bigram model? Why?

Write your answers below.

**Your prediction:**  
- Most frequent unigram:  
- Likely high-probability bigram:  
- Bigram vs trigram on unseen text:

## 3. Build N-gram counts

For an N-gram model, we count neighbouring sequences of words.

For example, in:

```text
language models predict words
```

the bigrams are:

```text
(<s>, language)
(language, models)
(models, predict)
(predict, words)
(words, </s>)
```

The trigram model uses sequences of three tokens.

We will use these counts to estimate conditional probabilities.

In [4]:
def add_boundaries(tokens, n):
    # TODO:
    # Bigram model: add one <s> and one </s>
    # Trigram model: add two <s> symbols and one </s>
    if n == 1:
          return list(tokens)
    return ["<s>"] * (n - 1) + list(tokens) + ["</s>"]


    pass

In [7]:
def make_ngrams(tokens, n):
    # TODO: return all adjacent n-grams as tuples
    # Example: ['a','b','c'], n=2 -> [('a','b'), ('b','c')]
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)- n+1)]
print(add_boundaries(["students","learn"],2))
print(add_boundaries(["students","learn"],3))
print(make_ngrams(["students","learn"],2))
print(make_ngrams(["students","learn"],3))
print(make_ngrams(["a","b","c"],1))
print(make_ngrams(["a","b","c"],2))
print(make_ngrams(["a","b","c"],3))



['<s>', 'students', 'learn', '</s>']
['<s>', '<s>', 'students', 'learn', '</s>']
[('students', 'learn')]
[]
[('a',), ('b',), ('c',)]
[('a', 'b'), ('b', 'c')]
[('a', 'b', 'c')]


In [13]:
# TODO: build unigram, bigram and trigram counts over the whole corpus.
# Suggested names:
# unigram_counts
# bigram_counts
# trigram_counts

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for sent in tokenized:
  unigram_counts.update(sent)
  bigram_counts.update(make_ngrams(add_boundaries(sent,2),2))
  trigram_counts.update(make_ngrams(add_boundaries(sent,3),3))

print(len(unigram_counts))
print(len(bigram_counts))
print(len(trigram_counts))

#print(unigram_counts)
#print(bigram_counts)
#print(trigram_counts)

15
28
32


### Inspect the most frequent N-grams
Create tables showing the 10 most frequent unigrams, bigrams and trigrams.

In [16]:
# TODO: display the 10 most frequent items from each Counter.

def top_table(counter, k = 10):
  return pd.DataFrame(counter.most_common(k), columns=['N-gram', 'count'])
display(top_table(unigram_counts))
display(top_table(bigram_counts))
display(top_table(trigram_counts))

,N-gram,count
0,language,6
1,models,5
2,students,3
3,learn,3
4,natural,2
5,processing,2
6,machine,2
7,learning,2
8,study,1
9,predict,1


,N-gram,count
0,"(language, models)",4
1,"(<s>, students)",3
2,"(students, learn)",2
3,"(natural, language)",2
4,"(language, processing)",2
5,"(machine, learning)",2
6,"(models, </s>)",2
7,"(<s>, language)",2
8,"(learn, natural)",1
9,"(processing, </s>)",1


,N-gram,count
0,"(<s>, <s>, students)",3
1,"(<s>, students, learn)",2
2,"(natural, language, processing)",2
3,"(language, models, </s>)",2
4,"(<s>, <s>, language)",2
5,"(<s>, language, models)",2
6,"(students, learn, natural)",1
7,"(learn, natural, language)",1
8,"(language, processing, </s>)",1
9,"(students, learn, machine)",1


## 4. Maximum-likelihood estimation of N-gram probabilities

For a bigram model,

$$
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
$$

For a trigram model,

$$
P(w_i \mid w_{i-2}, w_{i-1})
=
\frac{C(w_{i-2}, w_{i-1}, w_i)}
     {C(w_{i-2}, w_{i-1})}
$$

The key idea is simple: **among all occurrences of the history, how often was the next word the one we are asking about?**

This is a maximum-likelihood estimate because the probabilities are obtained directly from observed frequencies in the training corpus.

In [10]:
def bigram_probability(w1, w2, bigram_counts, unigram_context_counts):
    # TODO: implement P(w2 | w1)
    bigram_count = bigram_counts[(w1, w2)]
    context_count = unigram_context_counts[w1]
    if context_count == 0:
      return 0
    return bigram_count / context_count
    pass

def trigram_probability(w1, w2, w3, trigram_counts, bigram_context_counts):
    # TODO: implement P(w3 | w1,w2)
    trigram_count = trigram_counts[(w1,w2,w3)]
    context_count = bigram_context_counts[(w1,w2)]
    if context_count == 0:
      return 0
    return trigram_count / context_count
    pass

### Probability checks
Compute and interpret the following:

- `P(learn | students)`
- `P(models | language)`
- `P(language | learn)`
- `P(models | study, language)`

In [ ]:
# TODO: create the context counts needed by the probability functions,
# then compute the four probabilities above.

In [11]:
# 1. Create context counts from bigram and trigram counts
unigram_context_counts = Counter()
for (w1, w2), count in bigram_counts.items():
    unigram_context_counts[w1] += count

bigram_context_counts = Counter()
for (w1, w2, w3), count in trigram_counts.items():
    bigram_context_counts[(w1, w2)] += count

# 2. Compute the four probabilities
p1 = bigram_probability("students", "learn", bigram_counts, unigram_context_counts)
p2 = bigram_probability("language", "models", bigram_counts, unigram_context_counts)
p3 = bigram_probability("learn", "language", bigram_counts, unigram_context_counts)
p4 = trigram_probability("study", "language", "models", trigram_counts, bigram_context_counts)

# 3. Print the results
print(f"P(learn | students)         = {p1:.4f}")
print(f"P(models | language)        = {p2:.4f}")
print(f"P(language | learn)         = {p3:.4f}")
print(f"P(models | study, language) = {p4:.4f}")

P(learn | students)         = 0.6667
P(models | language)        = 0.6667
P(language | learn)         = 0.0000
P(models | study, language) = 1.0000


## 5. Sentence probability

Under a bigram model, the probability of a sentence is approximated using the Markov assumption:

$$
P(w_1,\ldots,w_m)
\approx
P(w_1\mid <s>)
\prod_{i=2}^{m}P(w_i\mid w_{i-1})
P(</s>\mid w_m)
$$

The trigram model conditions each word on the previous two tokens.

Because sentence probabilities are products of many values smaller than 1, they can become extremely small. In larger systems we therefore often work with **log probabilities**.

In [ ]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add boundaries
    # 3. multiply bigram probabilities
    pass

def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add trigram boundaries
    # 3. multiply trigram probabilities
    pass

In [17]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    # 1. Tokenize sentence into lowercase words
    tokens = tokenize(sentence)

    # 2. Add bigram boundaries (<s> at start, </s> at end)
    padded_tokens = add_boundaries(tokens, 2)

    # 3. Generate bigram pairs
    bigrams = make_ngrams(padded_tokens, 2)

    # 4. Multiply bigram probabilities
    prob = 1.0
    for w1, w2 in bigrams:
        p = bigram_probability(w1, w2, bigram_counts, context_counts)
        prob *= p
        if prob == 0.0:
            return 0.0  # Early exit if any bigram was never seen in training

    return prob


def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    # 1. Tokenize sentence into lowercase words
    tokens = tokenize(sentence)

    # 2. Add trigram boundaries (<s> <s> at start, </s> at end)
    padded_tokens = add_boundaries(tokens, 3)

    # 3. Generate trigram triplets
    trigrams = make_ngrams(padded_tokens, 3)

    # 4. Multiply trigram probabilities
    prob = 1.0
    for w1, w2, w3 in trigrams:
        p = trigram_probability(w1, w2, w3, trigram_counts, context_counts)
        prob *= p
        if prob == 0.0:
            return 0.0  # Early exit if any trigram was never seen in training

    return prob

### Compare sentence probabilities
Evaluate these two training-like sentences:

- `students learn machine learning`
- `language models assign probabilities`

Then explain why their probabilities differ.

In [ ]:
# TODO: compute bigram and trigram sentence probabilities for the two sentences.

In [18]:
sentences = [
    "students learn machine learning",
    "language models assign probabilities"
]

results = []
for s in sentences:
    p_bigram = bigram_sentence_probability(s, bigram_counts, unigram_context_counts)
    p_trigram = trigram_sentence_probability(s, trigram_counts, bigram_context_counts)
    results.append({
        "Sentence": s,
        "Bigram Probability": p_bigram,
        "Trigram Probability": p_trigram
    })

df_comparison = pd.DataFrame(results)
display(df_comparison)

,Sentence,Bigram Probability,Trigram Probability
0,students learn machine learning,0.047619,0.071429
1,language models assign probabilities,0.038095,0.071429


## 6. Evaluation on unseen text

A useful language model should perform well on text that was **not used to estimate its probabilities**.

We will use:

### Test sentence A
`students learn language models`

### Test sentence B
`language models learn patterns`

### Test sentence C
`students predict probabilities`

Before computing anything, inspect the corpus and predict which test sentences may contain unseen bigrams or trigrams.

### Prediction 2 — unseen sequences

Before running the next section, list at least one bigram/trigram that you think is unseen in each test sentence.

**A. students learn language models**  
Prediction:

**B. language models learn patterns**  
Prediction:

**C. students predict probabilities**  
Prediction:

In [20]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities"
]

def find_unseen_ngrams(sentence, n, ngram_counts):
    tokens = tokenize(sentence)
    padded_tokens = add_boundaries(tokens, n)
    ngrams = make_ngrams(padded_tokens, n)

    unseen = [gram for gram in ngrams if ngram_counts[gram] == 0]
    return unseen

# Display unseen N-grams for each test sentence
for s in test_sentences:
    unseen_bigrams = find_unseen_ngrams(s, 2, bigram_counts)
    unseen_trigrams = find_unseen_ngrams(s, 3, trigram_counts)

    print(f"--- Sentence: '{s}' ---")
    print(f"Unseen Bigrams  ({len(unseen_bigrams)}): {unseen_bigrams}")
    print(f"Unseen Trigrams ({len(unseen_trigrams)}): {unseen_trigrams}\n")

--- Sentence: 'students learn language models' ---
Unseen Bigrams  (1): [('learn', 'language')]
Unseen Trigrams (2): [('students', 'learn', 'language'), ('learn', 'language', 'models')]

--- Sentence: 'language models learn patterns' ---
Unseen Bigrams  (0): []
Unseen Trigrams (1): [('language', 'models', 'learn')]

--- Sentence: 'students predict probabilities' ---
Unseen Bigrams  (2): [('students', 'predict'), ('predict', 'probabilities')]
Unseen Trigrams (3): [('<s>', 'students', 'predict'), ('students', 'predict', 'probabilities'), ('predict', 'probabilities', '</s>')]



## 7. Perplexity

Perplexity is a standard intrinsic evaluation measure for language models.

For a sequence containing \(N\) predicted tokens,

$$
PP(W)
=
P(W)^{-1/N}
$$

Equivalently, using log probabilities,

$$
PP(W)
=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}\log P(w_i \mid \text{history})
\right)
$$

Interpretation:

- **lower perplexity** means the model assigns higher probability to the observed test sequence;
- **higher perplexity** means the sequence is more surprising to the model;
- if an unsmoothed model encounters an N-gram with probability zero, the perplexity becomes infinite.

Perplexity should only be compared meaningfully when models are evaluated on the **same tokenization and same test data**.

In [ ]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    # TODO:
    # Use log probabilities.
    # If any required bigram has probability 0, return math.inf.
    pass

def trigram_perplexity(sentence, trigram_counts, context_counts):
    # TODO:
    # Use log probabilities.
    # If any required trigram has probability 0, return math.inf.
    pass

In [21]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    tokens = tokenize(sentence)
    padded_tokens = add_boundaries(tokens, 2)
    bigrams = make_ngrams(padded_tokens, 2)

    N = len(bigrams)
    if N == 0:
        return math.inf

    log_prob_sum = 0.0
    for w1, w2 in bigrams:
        p = bigram_probability(w1, w2, bigram_counts, context_counts)
        if p == 0.0:
            return math.inf  # Zero probability yields infinite perplexity
        log_prob_sum += math.log(p)

    return math.exp(-log_prob_sum / N)


def trigram_perplexity(sentence, trigram_counts, context_counts):
    tokens = tokenize(sentence)
    padded_tokens = add_boundaries(tokens, 3)
    trigrams = make_ngrams(padded_tokens, 3)

    N = len(trigrams)
    if N == 0:
        return math.inf

    log_prob_sum = 0.0
    for w1, w2, w3 in trigrams:
        p = trigram_probability(w1, w2, w3, trigram_counts, context_counts)
        if p == 0.0:
            return math.inf  # Zero probability yields infinite perplexity
        log_prob_sum += math.log(p)

    return math.exp(-log_prob_sum / N)

In [ ]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

# TODO: construct a DataFrame with:
# sentence | bigram_perplexity | trigram_perplexity

In [22]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

results = []
for sentence in test_sentences:
    bp = bigram_perplexity(sentence, bigram_counts, unigram_context_counts)
    tp = trigram_perplexity(sentence, trigram_counts, bigram_context_counts)
    results.append({
        "sentence": sentence,
        "bigram_perplexity": bp,
        "trigram_perplexity": tp
    })

df_perplexity = pd.DataFrame(results)
display(df_perplexity)

,sentence,bigram_perplexity,trigram_perplexity
0,students learn language models,inf,inf
1,language models learn patterns,2.394694,inf
2,students predict probabilities,inf,inf


## 8. Diagnose the model
For every test sentence with infinite perplexity, print the first unseen N-gram that causes the failure.

In [ ]:
def first_unseen_ngram(sentence, n, ngram_counts):
    # TODO: return the first unseen n-gram, or None if every n-gram was seen.
    pass

# TODO: apply this to the test sentences for n=2 and n=3.

In [23]:
def first_unseen_ngram(sentence, n, ngram_counts):
    tokens = tokenize(sentence)
    padded_tokens = add_boundaries(tokens, n)
    ngrams = make_ngrams(padded_tokens, n)

    for gram in ngrams:
        if ngram_counts[gram] == 0:
            return gram  # Return the first unseen n-gram encountered

    return None  # All n-grams were seen


# Apply diagnostics to all test sentences for n=2 and n=3
diagnostics = []
for sentence in test_sentences:
    unseen_2gram = first_unseen_ngram(sentence, 2, bigram_counts)
    unseen_3gram = first_unseen_ngram(sentence, 3, trigram_counts)
    diagnostics.append({
        "sentence": sentence,
        "first_unseen_bigram": unseen_2gram,
        "first_unseen_trigram": unseen_3gram
    })

df_diagnostics = pd.DataFrame(diagnostics)
display(df_diagnostics)

,sentence,first_unseen_bigram,first_unseen_trigram
0,students learn language models,"(learn, language)","(students, learn, language)"
1,language models learn patterns,None,"(language, models, learn)"
2,students predict probabilities,"(students, predict)","(<s>, students, predict)"


## 9. Interpretation and error analysis

Answer in complete sentences.

1. Which model gave lower perplexity on the test sentences that both models could score?
2. Did the higher-order model always perform better? Explain using the size of this corpus.
3. Identify one unseen bigram and one unseen trigram from the test set.
4. Why does a zero probability create a serious problem when sentence probability is computed by multiplication?
5. What would you expect to happen if the training corpus became much larger?

## 10. Viva / discussion questions

1. Why is an N-gram language model called a probabilistic model?
2. What is the Markov assumption in a bigram model?
3. Why are sentence boundary symbols useful?
4. Why can a trigram model be more data-hungry than a bigram model?
5. Why is perplexity usually preferred over raw sentence probability when comparing sequences of different lengths?
6. Why can an unsmoothed N-gram model assign infinite perplexity to a perfectly grammatical sentence?